Import libraries and Load the required files


In [30]:
import numpy as np
import pandas as pd
from pathlib import Path

from IPython.display import display

print("Libraries imported successfully.")

Libraries imported successfully.


In [31]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "runs" / "models"
REPORT_DIR = PROJECT_ROOT / "runs" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

movies_df = pd.read_csv(PROJECT_ROOT / "data" / "ml-latest-small" / "movies.csv")

user_mapping = pd.read_csv(PROCESSED_DIR / "user_mapping.csv")
movie_mapping = pd.read_csv(PROCESSED_DIR / "movie_mapping.csv")

user_embeddings = np.load(MODEL_DIR / "user_embeddings.npy")
movie_embeddings = np.load(MODEL_DIR / "movie_embeddings.npy")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Movies shape:", movies_df.shape)

print("User embeddings:", user_embeddings.shape)
print("Movie embeddings:", movie_embeddings.shape)

display(test_df.head())

Train shape: (80419, 6)
Test shape: (10599, 6)
Movies shape: (9742, 3)
User embeddings: (610, 64)
Movie embeddings: (9742, 64)


,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,1270,5.0,2000-07-30 19:01:45+00:00,0,969
1,1,1240,5.0,2000-07-30 19:02:03+00:00,0,939
2,1,1206,5.0,2000-07-30 19:02:17+00:00,0,907
3,1,3702,5.0,2000-07-30 19:02:17+00:00,0,2764
4,1,3033,5.0,2000-07-30 19:02:42+00:00,0,2286


Prepare test relevant interactions

In [32]:
RELEVANCE_THRESHOLD = 4.0
K = 10

test_positive_df = test_df[
    test_df["rating"] >= RELEVANCE_THRESHOLD
].copy()

print("Positive test interactions:", len(test_positive_df))
print("Users with relevant test interactions:",
      test_positive_df["user_idx"].nunique())

display(test_positive_df.head())

Positive test interactions: 5058
Users with relevant test interactions: 574


,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,1270,5.0,2000-07-30 19:01:45+00:00,0,969
1,1,1240,5.0,2000-07-30 19:02:03+00:00,0,939
2,1,1206,5.0,2000-07-30 19:02:17+00:00,0,907
3,1,3702,5.0,2000-07-30 19:02:17+00:00,0,2764
4,1,3033,5.0,2000-07-30 19:02:42+00:00,0,2286


Prepare user-wise relevant movies

In [33]:
retrieval_train_df = pd.read_csv(
    PROCESSED_DIR / "retrieval_train.csv"
)

print("Retrieval training data loaded:", retrieval_train_df.shape)
display(retrieval_train_df.head())

trained_movie_indices = set(
    retrieval_train_df["movie_idx"].unique()
)

relevant_movies = (
    test_positive_df
    .groupby("user_idx")["movie_idx"]
    .apply(lambda x: set(x) & trained_movie_indices)
    .to_dict()
)

# Keep users who have at least one relevant movie available to the model
relevant_movies = {
    user_idx: movies
    for user_idx, movies in relevant_movies.items()
    if len(movies) > 0
}

print("Users eligible for warm-start evaluation:", len(relevant_movies))
print("Trained candidate movies:", len(trained_movie_indices))

Retrieval training data loaded: (39071, 7)


,userId,movieId,rating,timestamp,user_idx,movie_idx,label
0,1,804,4.0,2000-07-30 18:08:19+00:00,0,632,1
1,1,1210,5.0,2000-07-30 18:08:19+00:00,0,911,1
2,1,2018,5.0,2000-07-30 18:08:43+00:00,0,1493,1
3,1,2628,4.0,2000-07-30 18:08:43+00:00,0,1979,1
4,1,2826,4.0,2000-07-30 18:08:43+00:00,0,2126,1


Users eligible for warm-start evaluation: 565
Trained candidate movies: 5320


Exclude movies already seen during training

This ensures that our evaluation focuses on recommending movies that the user has not already interacted with in the training dataset.

In [34]:
seen_movies = (
    train_df
    .groupby("user_idx")["movie_idx"]
    .apply(set)
    .to_dict()
)

print("Users with training history:", len(seen_movies))

Users with training history: 610


Generate Top-10 recommendations

Calculating similarity between each user embedding and all movie embeddings using the dot product.

In [35]:
trained_movie_indices_array = np.array(
    sorted(trained_movie_indices)
)

def get_top_k_recommendations(user_idx, k=10):

    user_vector = user_embeddings[user_idx]

    candidate_embeddings = movie_embeddings[
        trained_movie_indices_array
    ]

    scores = np.dot(candidate_embeddings, user_vector)

    user_seen = seen_movies.get(user_idx, set())

    seen_mask = np.isin(
        trained_movie_indices_array,
        list(user_seen)
    )

    scores[seen_mask] = -np.inf

    valid_count = np.sum(np.isfinite(scores))

    if valid_count == 0:
        return []

    k_actual = min(k, valid_count)

    top_positions = np.argpartition(
        scores,
        -k_actual
    )[-k_actual:]

    top_positions = top_positions[
        np.argsort(scores[top_positions])[::-1]
    ]

    recommended_movie_indices = (
        trained_movie_indices_array[top_positions]
    )

    return recommended_movie_indices.tolist()

In [36]:
sample_user = list(relevant_movies.keys())[0]

sample_recommendations = get_top_k_recommendations(sample_user, K)

print("Sample User ID:", sample_user)
print("Recommended Movie Indices:", sample_recommendations)

Sample User ID: 0
Recommended Movie Indices: [2173, 1192, 2684, 1288, 2770, 2652, 768, 1125, 8758, 1143]


Calculate Recall@10 and NDCG@10

In [37]:
def calculate_recall_at_k(recommended, relevant, k=10):
    recommended_k = recommended[:k]

    hits = len(set(recommended_k) & relevant)

    return hits / len(relevant) if relevant else 0.0


def calculate_ndcg_at_k(recommended, relevant, k=10):
    recommended_k = recommended[:k]

    dcg = 0.0

    for rank, movie_idx in enumerate(recommended_k, start=1):
        if movie_idx in relevant:
            dcg += 1.0 / np.log2(rank + 1)

    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg if idcg > 0 else 0.0

Evaluate the Two-Tower model

In [38]:
recall_scores = []
ndcg_scores = []

recommendations_by_user = {}

for user_idx, relevant in relevant_movies.items():

    recommendations = get_top_k_recommendations(user_idx, K)

    recommendations_by_user[user_idx] = recommendations

    recall = calculate_recall_at_k(
        recommendations,
        relevant,
        K
    )

    ndcg = calculate_ndcg_at_k(
        recommendations,
        relevant,
        K
    )

    recall_scores.append(recall)
    ndcg_scores.append(ndcg)


two_tower_recall = np.mean(recall_scores)
two_tower_ndcg = np.mean(ndcg_scores)

print("Two-Tower Retrieval Model Results")
print("--------------------------------")
print(f"Recall@{K}: {two_tower_recall:.4f}")
print(f"NDCG@{K}:   {two_tower_ndcg:.4f}")

Two-Tower Retrieval Model Results
--------------------------------
Recall@10: 0.0158
NDCG@10:   0.0138


Inspect mapping columns

In [39]:
print("User mapping columns:", user_mapping.columns.tolist())
print("Movie mapping columns:", movie_mapping.columns.tolist())

display(movie_mapping.head())
display(movies_df.head())

User mapping columns: ['userId', 'user_idx']
Movie mapping columns: ['movieId', 'movie_idx']


,movieId,movie_idx
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Display actual movie recommendations

This converts recommended movie indices into movie titles.

In [40]:
movie_id_col = next(
    col for col in movie_mapping.columns
    if col.lower() in ["movieid", "movie_id"]
)

movie_idx_col = next(
    col for col in movie_mapping.columns
    if col.lower() == "movie_idx"
)

movie_lookup = movie_mapping.merge(
    movies_df,
    left_on=movie_id_col,
    right_on="movieId",
    how="left"
)

sample_user = list(recommendations_by_user.keys())[0]

sample_movie_indices = recommendations_by_user[sample_user]

sample_recommendations_df = pd.DataFrame({
    "movie_idx": sample_movie_indices,
    "rank": range(1, K + 1)
})

sample_recommendations_df = sample_recommendations_df.merge(
    movie_lookup,
    on="movie_idx",
    how="left"
)

display(
    sample_recommendations_df[
        ["rank", "title", "genres"]
    ]
)

,rank,title,genres
0,1,"Mystery, Alaska (1999)",Comedy|Drama
1,2,Cop Land (1997),Action|Crime|Drama|Thriller
2,3,Center Stage (2000),Drama|Musical
3,4,Scream 2 (1997),Comedy|Horror|Mystery|Thriller
4,5,Firestarter (1984),Horror|Thriller
5,6,The Hunger (1983),Horror
6,7,"Love Bug, The (1969)",Children|Comedy
7,8,Booty Call (1997),Comedy|Romance
8,9,Patton Oswalt: Tragedy Plus Comedy Equals Time...,Comedy
9,10,Double Team (1997),Action


Compare with baseline models

In [41]:
comparison_df = pd.DataFrame({
    "Model": [
        "Popularity-Based",
        "User-Based Collaborative Filtering",
        "Two-Tower Retrieval"
    ],
    "Recall@10": [
        0.0426,
        0.0666,
        two_tower_recall
    ],
    "NDCG@10": [
        0.0438,
        0.0637,
        two_tower_ndcg
    ]
})

display(comparison_df)

comparison_df.to_csv(
    REPORT_DIR / "two_tower_evaluation.csv",
    index=False
)

print("Evaluation results saved successfully.")

,Model,Recall@10,NDCG@10
0,Popularity-Based,0.042600,0.043800
1,User-Based Collaborative Filtering,0.066600,0.063700
2,Two-Tower Retrieval,0.015801,0.013756


Evaluation results saved successfully.


Save recommendation examples

In [42]:
recommendation_rows = []

for user_idx, movie_indices in recommendations_by_user.items():

    for rank, movie_idx in enumerate(movie_indices, start=1):

        recommendation_rows.append({
            "user_idx": user_idx,
            "rank": rank,
            "movie_idx": movie_idx
        })

recommendations_df = pd.DataFrame(recommendation_rows)

recommendations_df.to_csv(
    REPORT_DIR / "two_tower_recommendations.csv",
    index=False
)

print("Recommendation examples saved.")
print("Total recommendations:", len(recommendations_df))

Recommendation examples saved.
Total recommendations: 5650
